RAG PIPELINE

In [4]:
## Data Ingestion to vector DB Pipeline

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

Loading the Documents


In [5]:
## to read all the text files

def read_all_txts(txts_directory):
    all_docs =[]
    txt_dir = Path(txts_directory)

    # find all txt files 
    txt_files = list(txt_dir.glob("**/*.txt"))
    print(f"Found {len(txt_files)} text files in the directory")

    for txt_file in txt_files:
        print(f"\nProcessing: {txt_file.name}")
        try:
            loader = TextLoader(str(txt_file))
            docs = loader.load()

            # add source info to metadata
            for doc in docs:
                doc.metadata['source_file'] = txt_file.name
                doc.metadata['file_type'] = 'txt'

            all_docs.extend(docs)
            print(f"Loaded {len(docs)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

# Process the all text file in the data directory
all_txt_docs = read_all_txts("../data")

Found 4 text files in the directory

Processing: dl_intro.txt
Loaded 1 pages

Processing: nn_intro.txt
Loaded 1 pages

Processing: ml_intro.txt
Loaded 1 pages

Processing: nlp_intro.txt
Loaded 1 pages

Total documents loaded: 4


Converting into the chunks

In [6]:
# Text spliting into chunks
def split_documents(docs, chunk_size=500, chunk_overlap =50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ","",]
    )

    splited_docs = text_splitter.split_documents(docs)
    print(f"Split {len(docs)} documents into {len(splited_docs)} chunks.")

    return splited_docs

In [7]:
chunks = split_documents(all_txt_docs)

Split 4 documents into 43 chunks.


Embeddings and model loading

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid, os
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple

In [9]:
class EmbeddingManager:
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args: 
            model_name: HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)

        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise


    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """ 
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager
        

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8116.74it/s]


Creating VectorDB

In [ ]:
class VectorStore:
    """Manages the documents embeddings in a Vector store."""

    def __init__(self, collection_name: str = "txt_documents", presist_directory:str="../data/vector_store"):
        """ 
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            presist_directory: Directory to presist the vector store
        """
        self.collection_name=collection_name
        self.presist_directory = presist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        

    def _initialize_store(self):

        try:
            #create presistent ChromaDb client
            os.makedirs(self.presist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.presist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Text document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")


        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for chromaDb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_len']=len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfullly added {len(documents)} documents to the vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

# Initialize it
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: txt_documents
Existing documents in collection: 129


converting chunks into embeddings

In [11]:
texts=[doc.page_content for doc in chunks] #get the whole text from our document

# generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in vectordb
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 43 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  6.29it/s]

Generated embeddings with shape: (43, 384)
Adding 43 documents to vector store...
Successfullly added 43 documents to the vector store
Total documents in collection: 172


In [12]:
class RagRetriever:

    def __init__(self,vectorstore:VectorStore, embedding_manager: EmbeddingManager):
        """ 
        Initialize the retriever
        
        Args:
            vectorstore: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vectorstore= vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k:int=5, score_threshold: float =0.0) -> List[Dict[str,Any]]:
        """ 
        retrieve relevant documents for a query
        
        args:
            query: the search query
            top_k: Number of top result to return
            score_threshold: minimum similarity score threshold
            
        returns:
            list of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top-K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_emb = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_emb.tolist()],
                n_results=top_k
            ) 

            # process result
            retrieved_docs =[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert the distance to similarity score
                    similarity_score = 1- distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance': distance,
                            'rank':i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print(f"No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RagRetriever(vectorstore,embedding_manager)
rag_retriever 

In [13]:
rag_retriever.retrieve("what is machine learning")

Retrieving documents for query: 'what is machine learning'
Top-K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.98it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_2635a47a_15',
  'content': 'Machine learning is widely used in real-world applications such as recommendation systems, spam email filtering, self-driving cars, facial recognition, medical diagnosis, chatbots, and language translation. The main goal of machine learning is to create systems that can improve their performance automatically with experience. There are three major types of machine learning: supervised learning, where models learn from labeled data; unsupervised learning, where models identify hidden patterns in',
  'metadata': {'content_len': 499,
   'doc_index': 15,
   'source': '../data/text_files/ml_intro.txt',
   'file_type': 'txt',
   'source_file': 'ml_intro.txt'},
  'similarity_score': 0.3946094512939453,
  'distance': 0.6053905487060547,
  'rank': 1},
 {'id': 'doc_e4b226c1_15',
  'content': 'Machine learning is widely used in real-world applications such as recommendation systems, spam email filtering, self-driving cars, facial recognition, medical diagn

Integration VectorDB Pipeline with LLM output


In [23]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os
groq_api_key = os.environ['GROQ_API_KEY']

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# simple RAG function
def rag_simple(query,retriever,llm,top_k=3):
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    #generate the answer from GROQ LLM

    prompt=f"""Use the following context to answer the question concisely.
        context: 
        {context}
        question:
        {query}"
        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [24]:
answer = rag_simple("what is neural network?",rag_retriever,llm)
answer

Retrieving documents for query: 'what is neural network?'
Top-K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


'A neural network is a computer system inspired by the human brain, consisting of layers of interconnected nodes (neurons) that work together to learn patterns from data.'